# Voice Recognition Lab 1: SVD/PCA Classification

### EECS 16A: Foundations of Signals, Dynamical Systems, and Information Processing, Spring 2026


Junha Kim, Jessica Fan, Savit Bhat, Jack Kang (Fall 2024).

Sonia Chacon (Spring 2025)

Andrew Song (Fall 2025)


This lab was heavily inspired by previous EECS16B lab 8, written by Nathaniel Mailoa, Emily Naviasky, et al.



### Lab 1: SVD/PCA

* [Task 1: Data Collection](#part1)
* [Task 2: Data Preprocessing](#part2)
* [Task 3: PCA via SVD](#task3)
* [Task 4: Clustering Data Points](#task4)
* [Task 5: Testing your Classifier](#task5)
* [Task 6: Live Testing](#task6)

<a id='intro'></a>
## <span style="color:navy">Introduction</span>

**<span style="color:red">PLEASE READ: THERE IS AN IN-PERSON COMPONENT TO THIS LAB. PLEASE PLAN TO ARRIVE AT CORY 111 AT LEAST 30 MINUTES BEFORE YOUR SECTION ENDS.</span>** 

Throughout lectures and discussion, you have learned theoretical concepts of the DFT and SVD. In the voice recognition lab module, we'll be applying these concepts from class to build an audio classifier using Mel-Scaled Short-Time Fourier transform (STFT) and Principal Component Analysis (PCA), an application of the Singular Value Decomposition (SVD).

This lab takes in voice recordings, sampling the waveform of an analog signal into discrete points. Multiple syllables will generate multiple peaks and troughs, and soft syllables will have discrete points of lower magnitude than hard syllables (eg. words like "here" and "pear" will have very similar voice recordings)

In this module, you will build a voice classifier for words of different syllables and intonation in two parts:
- In lab 1, we will explore dimensionality reduction and classification of voice signals using PCA.
- In lab 2, we will extend our previous scheme by preprocessing our audio signal with the STFT (very similar to spectrograms that you have seen in the Shazam lab) and feed that as an input to our previous PCA classifier.

### Overview of Classification Procedure
Below is an overview of the classification procedure we will be following in this lab.
1. Collect recordings of different words. This will form our data set. (This is optional and can be skipped)
2. Split the data into 2 sets: training and testing data
3. Preprocess the data to align the words.
4. Perform SVD and evaluate on the training data split.
5. Classify each word using clustering in the PCA basis.
6. Evaluate performance of your PCA model by running the model on testing data.
7. Make sure you (and your GSI) are satisfied with the classifier's accuracy.

### Side Note: Datasets in Machine Learning Applications
It is common practice in machine learning applications to split a dataset into a training set and testing set (some common ratios for train:test are 80:20 or 70:30). In this lab, we will split 70:30.

In [ ]:
import sounddevice
import pyaudio
from tqdm.notebook import trange
from IPython import display
import wave
import numpy as np
import csv
import time
import matplotlib.pyplot as plt
import scipy.io
import utils
from mpl_toolkits.mplot3d import Axes3D
cm = ['blue', 'red', 'green', 'orange', 'black', 'purple']
%matplotlib inline

<a id='part1'></a>
## <span style="color:navy">Task 1: Data Collection (OPTIONAL)</span>
**This part of the section requires you to record at the lab station. This section is optional.**

We have provided data for 4 words for the sake of time. To be involved with the procedure, we ask that you record a set of 40 recordings for one word of your choosing.

When humans distinguish words, they listen for temporal and frequency differences to determine what is being said. Let's keep that in mind when we choose our words, as we will eventually be performing a frequency transform pre-classification in the Mel-Scaled STFT.

When you think of speech signals, you might notice that each word/sound has a distinctive shape. Taking the shape of the magnitude of a signal is called enveloping, exemplified in the plot below. We want to do some filtering to retrieve the envelope of the audio signal, which we will use for calculating the SVD.

**Given this information, brainstorm one word for your recordings.**

### Voice Recordings

Now we will record **40** audio samples for one word. We recommend that you split the recording between partners to maximize variance between the recorded signals. For your word, make sure to note who said it and how it was said (like through a video/audio recording on your phone) so that it is easier to reproduce later for live classification.

Please read the full instructions for these tasks before proceeding.
**For each chosen word, do the following:**
1. Run the cell below. Add the file name
    - The cell will ask for a filename to your csv file recording. The purpose of this csv file is to preserve your recordings as a permanent file, rather than as a temporary variable in jupyter notebook. This allows you to reuse the recordings you made in case you decide to take a break between the lab, or accidentally restart your jupyter notebook. 
    - We recommend naming your csv file to: `YOUR_WORD.csv`
2. Begin Recording. Each time you press enter, say your word once within the 3 second interval.
    - **Pronounce the word consistently**
    - "Good" audio data has a high signal to noise ratio (SNR). Recording words while far away from the microphone may cause your intended word to blend in with background noise. However, speaking too loudly and/or too closely into the mic) may also cause your output to rail. Here's what a railing (clipping) audio signal may look like:
    ![clippingsignal](clippingsignal.png)
    - You can see that this signal is utilizing almost the maximum magnitude of signal allowed in this audio file format for the majority of the duration. When an audio signal is too loud, its magnitude will exceed the maximum allowed value and "rail" to that value. We don't want this--please re-record further away from your mic if this happens.
    - **We recommend that after taking 3 or 4 recordings for the first time, stop the program by entering "stop" in the text box popup and check `YOUR_WORD.csv` make sure that it looks like a sound wave as opposed to being full of super low values. *It might help to graph the data as a line plot in Excel.*** You don't want to record 40 times only to find that your recordings weren't working.
    - Type "d" to delete the most recent recording
    - repeat to make 40 recording of each word.

### Before moving on, please note that:

You may realize in the next section that one or two of your words are not sorting quite as well as you would like. Don't be afraid to come back to this section and try collecting different words based on what you have learned makes a word sortable.

In [ ]:
# OPTIONAL
# If you would like to run this cell. Please uncomment the last line.

# run the recording script
# utils.create_recording_csv()

<a id='part2'></a>
## <span style="color:navy">Task 2: Data Preprocessing</span>

Different recordings of the same word can look wildly different, depending on factors like when you started saying the word and how quickly you said it (assuming you are not a robot that can repeat the word 40 times exactly the same way). Thus, before we can use the recorded data for PCA, we must first process the data. We will do this according to the following procedure:
1. Split the dataset into a training dataset and a test dataset
2. Align the audio recordings
3. Apply an envelope (low pass filter) to the recordings to remove irrelevant high-frequency components

In [ ]:
# YOUR CODE HERE: fill in the blank for the word you just recorded. 
# or remove the empty string if you did not record a new word
all_words_arr = ['jack', 'jason', 'jessica', 'entanglement', '']

### Task 2a: Align Audio Recordings
Let's begin by splitting our data into a training and testing set with a 70/30 split. Run the code below to do so.

In [ ]:
# Load data from csv
train_test_split_ratio = 0.7
train_dict = {}
test_dict = {}

# Build the dictionary of train and test samples.
for i in range(len(all_words_arr)):
    word_raw = utils.read_csv("{}.csv".format(all_words_arr[i]))
    word_raw_train, word_raw_test = utils.train_test_split(word_raw, train_test_split_ratio)
    train_dict[all_words_arr[i]] = word_raw_train
    test_dict[all_words_arr[i]] = word_raw_test

# Count the minimum number of samples you took across the six recorded words. These variables might be useful for you!
num_samples_train = min(list(map(lambda x : np.shape(x)[0], train_dict.values())))
num_samples_test = min(list(map(lambda x : np.shape(x)[0], test_dict.values())))

# Crop the number of samples for each word to the minimum number so all words have the same number of samples.
for key, raw_word in train_dict.items():
    train_dict[key] = raw_word[:num_samples_train,:]

for key, raw_word in test_dict.items():
    test_dict[key] = raw_word[:num_samples_test,:]

Let's take a look at all the training samples.

**<span style="color:red">Important: It's okay if the recordings aren't aligned. The code in the next part will align the data.</span>** 

In [ ]:
# Plot all training samples
word_number = 0
selected_words_arr = all_words_arr
for word_raw_train in train_dict.values():
    plt.plot(word_raw_train.T)
    plt.title('Training sample for "{}"'.format(selected_words_arr[word_number]))
    word_number += 1
    plt.show()

### Task 2b: Align Audio Recordings

As seen above, the speech is a fraction of the 3 second window, and each sample starts at different times. PCA is not good at interpreting delay, so we need to align the recordings and trim to a smaller segment of the sample where the speech is present. To do this, we will use a thresholding algorithm.

First, we define a **`threshold`** relative to the maximum value of the data. We say that any signal that crosses the threshold is the start of a speech command. In order to not lose the first couple samples of the speech command, we say that the command starts **`pre_length`** samples _before_ the threshold is crossed. We then use a window of the data that is **`length`** long, and try to capture the entire command in that window.

**Play around with the parameters `length`, `pre_length` and `threshold`** in the cells below to find appropriate values corresponding to your voice and chosen commands. You should see the results and how much of your command you captured in the plots generated below.

We also pass in a `envelope=True` argument to our `process_data` function: this filters out the frequencies higher than our vocal range, to get back an "envelope" of the waveform of our voice recordings. Why might this be useful?

Also note that we're "normalizing" the signal towards the end--this is very common in a lot of signal processing applications; another name for this is "min-max feature scaling," where rescale the amplitude range of the signal to be in $[0,1]$ by dividing the signal by the maximum absolute value. This is different from taking a norm of a vector as we've seen in class.

In [ ]:
def align_recording(recording, length, pre_length, threshold, envelope=False):
    """
    align a single audio samples according to the given parameters.
    
    Args:
        recording (np.ndarray): a single audio sample.
        length (int): The length of each aligned audio snippet.
        pre_length (int): The number of samples to include before the threshold is first crossed.
        threshold (float): Used to find the start of the speech command. The speech command begins where the
            magnitude of the audio sample is greater than (threshold * max(samples)).
        envelope (bool): if True, use enveloping.
    
    Returns:
        aligned recording.
    """
    
    if envelope:
        recording = utils.envelope(recording, 5400, 100)
    
    # Find the threshold
    recording_threshold = threshold * np.max(recording)

    # TODO: Use recording_threshold, length, and prelength to cut the snippet to the correct length
    # we note where the recording magnitude first exceeds recording_threshold.
    # then, we leave prelength number of samples before the crossing of the threshold, which is where our recording starts.
    # we cut the recording so that it's 'length' samples away from the start of the recording.
    
    i = pre_length
    while ... : # YOUR CODE HERE
        i += 1

    snippet_start = np.clip(i - pre_length, a_min=0, a_max=len(recording) - length)
    snippet = recording[ ... ] # YOUR CODE HERE

    # TODO: Normalize the recording.
    # "Normalize" in our case is dividing the signal by the maximum absolute value (different from taking the norm of a vector)
    snippet_normalized = # YOUR CODE HERE
    
    return snippet_normalized

In [ ]:
def align_data(data, length, pre_length, threshold, envelope=False):
    """
    align all audio samples in dataset. (apply align_recording to all rows of the data matrix)
    
    Args:
        data (np.ndarray): Matrix where each row corresponds to a recording's audio samples.
        length (int): The length of each aligned audio snippet.
        pre_length (int): The number of samples to include before the threshold is first crossed.
        threshold (float): Used to find the start of the speech command. The speech command begins where the
            magnitude of the audio sample is greater than (threshold * max(samples)).
    
    Returns:
        Matrix of aligned recordings.
    """
    assert isinstance(data, np.ndarray) and len(data.shape) == 2, "'data' must be a 2D matrix"
    assert isinstance(length, int) and length > 0, "'length' of snippet must be an integer greater than 0"
    assert 0 <= threshold <= 1, "'threshold' must be between 0 and 1"
    snippets = []

    # Iterate over the rows in data
    for recording in data:
        snippets.append(align_recording(recording, length, pre_length, threshold, envelope))

    return np.vstack(snippets)

In [ ]:
def process_data(dict_raw, length, pre_length, threshold, plot=True, envelope=False):
    """
    Process the raw data given parameters and return it. (wrapper function for align_data)
    
    Args:
        dict_raw (np.ndarray): dictionary of all words: data matrix.
        length (int): The length of each aligned audio snippet.
        pre_length (int): The number of samples to include before the threshold is first crossed.
        threshold (float): Used to find the start of the speech command. The speech command begins where the
            magnitude of the audio sample is greater than (threshold * max(samples)).
        plot (boolean): Plot the dataset if true.
            
    Returns:
        Processed data dictionary.
    """
    processed_dict = {}
    word_number = 0
    for key, word_raw in dict_raw.items():
        word_processed = align_data(word_raw, length, pre_length, threshold, envelope=envelope)
        processed_dict[key] = word_processed
        if plot:
            plt.plot(word_processed.T)
            plt.title('Samples for "{}"'.format(selected_words_arr[word_number]))
            word_number += 1
            plt.show()
            
    return processed_dict

In [ ]:
# TODO: Edit the parameters to get the best alignment.
length = # YOUR CODE HERE
pre_length = 400 # Modify this as necessary
threshold = # YOUR CODE HERE

processed_train_dict = process_data(train_dict, length, pre_length, threshold, envelope=True)

You should now notice the improved alignment for the samples. Can you tell which word is which just by the envelope? If any of your words look too similar to each another, then PCA will likely have a difficult time distinguishing between them, but the STFT version of PCA might circumvent this issue. Why is that?

<a id='task3'></a>
## <span style="color:navy">Task 3: PCA via SVD</span>

### SVD/PCA Resources
- http://www.ams.org/publicoutreach/feature-column/fcarc-svd
- https://stats.stackexchange.com/questions/2691/making-sense-of-principal-component-analysis-eigenvectors-eigenvalues
- https://towardsdatascience.com/pca-and-svd-explained-with-numpy-5d13b0d2a4d8

### Task 3a: Generate and Preprocess PCA Matrix

Now that we have our aligned data, we will build the PCA input matrix from that data by stacking all the data vertically.

In [ ]:
processed_A = np.vstack(list(processed_train_dict.values()))

The first step of PCA is to center the data's mean at zero and store it in demeaned_A. Please note that you want to get the mean of each feature (what are the features?). The function np.mean might be helpful here, along with specifying the axis parameter.

In [ ]:
# Zero-mean the matrix A
mean_vec = # YOUR CODE HERE
demeaned_A = # YOUR CODE HERE
print(processed_A.shape)
print(mean_vec.shape)

### Task 3b: PCA

Take the SVD of your demeaned data - `np.linalg.svd` may be useful here.

In [ ]:
# Take the SVD of matrix demeaned_A
U, S, Vt = # YOUR CODE HERE

Visually inspect your sigma values. They should tell you how many principal components you need.

In [ ]:
# Plot out the sigma values (Hint: Use plt.stem for a stem plot)
# YOUR CODE HERE

### Task 3c: Choosing a Basis using Principal Components

Set the `new_basis` argument to be a basis of the first 3 principal components.

Hint 1: Of the three outputs from the SVD function call, which one will contain the principal components onto which we want to project our data points? Do we need to transpose it?

Hint 2: For $A \in \mathbb{R}^{n \times n}$, $U$ contains the basis of the column space of $A$, and $V$ contains the basis of the row space of $A$.

Sanity check: When you plot `new_basis` you should see a number of line plots equal to the number of principal components you've chosen. You should have 3 differently colored components, each of length ‘length’ that you chose above.

**NOTE: The projection onto a subspace with orthogonal basis vectors (i.e. a matrix with orthogonal columns) reduces to an inner product:**

$$\text{proj}_{\text{Col(Q)}} (\vec{y}) = QQ^T \vec{y} = \sum^n_{i=1} \langle \vec{y}, \vec{q}_i \rangle \vec{q}_i$$

refer to eecs16b's [Spring 2024 note 13](https://eecs16b.org/notes/sp24/note13.pdf) for a proof of this.

In [ ]:
# Plot the principal component(s)
new_basis = # YOUR CODE HERE
plt.plot(new_basis)
plt.show()

Now project the data in the matrix A onto the new basis and plot it. For three principal components, in addition to the 3D plot, we also provided 2D plots which correspond to the top and side views of the 3D plot. Do you see clustering? Do you think you can separate the data easily?

In [ ]:
# Project the data onto the new basis
# Hint: np.dot() may help, as well as printing the dimensions.
proj = # YOUR CODE HERE

if new_basis.shape[1] == 3:
    fig=plt.figure(figsize=(10,5))
    ax = fig.add_subplot(111, projection='3d')
    for i in range(len(all_words_arr)):
        Axes3D.scatter(ax, *proj[i*num_samples_train:num_samples_train*(i+1)].T, c=cm[i], marker = 'o', s=20)
    plt.legend(all_words_arr,loc='center left', bbox_to_anchor=(1.07, 0.5))
    
    fig, axs = plt.subplots(1, 3, figsize=(15,5))
    for i in range(len(all_words_arr)):
        axs[0].scatter(proj[i*num_samples_train:num_samples_train*(i+1),0], proj[i*num_samples_train:num_samples_train*(i+1),1], c=cm[i], edgecolor='none')
        axs[1].scatter(proj[i*num_samples_train:num_samples_train*(i+1),0], proj[i*num_samples_train:num_samples_train*(i+1),2], c=cm[i], edgecolor='none')
        axs[2].scatter(proj[i*num_samples_train:num_samples_train*(i+1),1], proj[i*num_samples_train:num_samples_train*(i+1),2], c=cm[i], edgecolor='none')
    axs[0].set_title("View 1")
    axs[1].set_title("View 2")
    axs[2].set_title("View 3")
    plt.legend(all_words_arr,loc='center left', bbox_to_anchor=(1, 0.5))
    plt.show()
    
elif new_basis.shape[1] == 2:
    fig=plt.figure(figsize=(10,5))
    for i in range(len(all_words_arr)):
        plt.scatter(proj[i*num_samples_train:num_samples_train*(i+1),0], proj[i*num_samples_train:num_samples_train*(i+1),1], edgecolor='none')

    plt.legend(all_words_arr,loc='center left', bbox_to_anchor=(1, 0.5))
    plt.show()

Like in many AI applications, the data above are noisy, so we expect some classification errors. The important part is that you see strong clustering of your words. 

If you don't see clustering, try to think about why this might be the case. Things you might want to ask yourself:
- How does PCA create the clusters? 
- Which characteristics of your waveform will PCA favor when clustering? 
- How can you choose your words to maximize the differences between the classes?

Once you think you have decent clustering, move on to automating classification.

<a id='task4'></a>
## <span style="color:navy">Task 4: Clustering Data Points</span>

#### Implement `find_centroids`, which finds the center of each cluster.

In [ ]:
def find_centroid(clustered_data):
    """Find the center of each cluster by taking the mean of all points in a cluster.
    It may be helpful to recall how you constructed the data matrix (e.g. which rows correspond to which word)
    
    Parameters:
        clustered_data (np.array): the data already projected onto the new basis (for one word)
        
    Returns: 
        centroid: The centroid of the cluster
    """
    
    return # YOUR CODE HERE

In [ ]:
# run find_centroid on each word
centroids = []
for i in range(len(all_words_arr)):
    centroid = find_centroid(proj[i*num_samples_train:(i + 1)* num_samples_train])
    centroids.append(centroid)

print(centroids)

Run the cell below to plot your centroids along with your projected data.

In [ ]:
centroid_list = np.vstack(centroids)
colors = cm[:(len(centroids))]

for i, centroid in enumerate(centroid_list):
    print('Centroid {} is at: {}'.format(i, str(centroid)))

if new_basis.shape[1] == 3:
    fig=plt.figure(figsize=(10,7))
    ax = fig.add_subplot(111, projection='3d')
    for i in range(len(selected_words_arr)):
        Axes3D.scatter(ax, *proj[i*num_samples_train:num_samples_train*(i+1)].T, c=cm[i], marker = 'o', s=20)
    plt.legend(selected_words_arr, loc='center left', bbox_to_anchor=(1.07, 0.5))
    for i in range(len(selected_words_arr)):
        Axes3D.scatter(ax, *np.array([centroids[i]]).T, c=cm[i], marker = '*', s=300)
    plt.title("Training Data")
    
    fig, axs = plt.subplots(1, 3, figsize=(15,5))
    for i in range(len(all_words_arr)):
        axs[0].scatter(proj[i*num_samples_train:num_samples_train*(i+1),0], proj[i*num_samples_train:num_samples_train*(i+1),1], c=cm[i], edgecolor='none')
        axs[1].scatter(proj[i*num_samples_train:num_samples_train*(i+1),0], proj[i*num_samples_train:num_samples_train*(i+1),2], c=cm[i], edgecolor='none')
        axs[2].scatter(proj[i*num_samples_train:num_samples_train*(i+1),1], proj[i*num_samples_train:num_samples_train*(i+1),2], c=cm[i], edgecolor='none')
    axs[0].set_title("View 1")
    axs[1].set_title("View 2")
    axs[2].set_title("View 3")
    plt.legend(all_words_arr, loc='center left', bbox_to_anchor=(1, 0.5))
    axs[0].scatter(centroid_list[:,0], centroid_list[:,1], c=colors, marker='*', s=300)
    axs[1].scatter(centroid_list[:,0], centroid_list[:,2], c=colors, marker='*', s=300)
    axs[2].scatter(centroid_list[:,1], centroid_list[:,2], c=colors, marker='*', s=300)
    plt.show()

elif new_basis.shape[1] == 2:
    fig=plt.figure(figsize=(10,7))
    for i in range(len(all_words_arr)):
        plt.scatter(proj[i*num_samples_train:num_samples_train*(i+1),0], proj[i*num_samples_train:num_samples_train*(i+1),1], c=colors[i], edgecolor='none')

    plt.scatter(centroid_list[:,0], centroid_list[:,1], c=colors, marker='*', s=300)
    plt.legend(all_words_arr,loc='center left', bbox_to_anchor=(1, 0.5))
    plt.title("Training Data")
    plt.show()

<a id='task5'></a>
## <span style="color:navy">Task 5: Testing your Classifier</span>

Great! Now that we have the means (centroid) for each word, let's evaluate performance. Recall that we will classify each data point according to the centroid with the least Euclidian distance to it.

Before we perform classification, we need to do the same preprocessing to the test data that we did to the training data (enveloping, demeaning, projecting onto the PCA basis). You have already written most of the code for this part. However, note the difference in variable names as we are now working with test data.

First let's look at what our raw test data looks like.

### Task 5a: Test Data Preprocessing

Let's repeat what we did for our training data, so we can try out the test data on our trained model!

In [ ]:
# Plot all test samples
word_number = 0
for word_raw_test in test_dict.values():
    plt.plot(word_raw_test.T)
    plt.title('Test sample for "{}"'.format(all_words_arr[word_number]))
    word_number += 1
    plt.show()

Perform enveloping and trimming of our test data.

In [ ]:
processed_test_dict = process_data(test_dict, length, pre_length, threshold, plot=True, envelope=True)

Construct the PCA matrix. Refer to the code we used above for the training set!

In [ ]:
processed_A_test = # YOUR CODE HERE

**Now we will do something slightly different.**

Previously, you projected data onto your PCA basis with $ (x - \bar{x})P $, where $\bar{x}$ is the mean vector, $x$ is a single row of `processed_A`, and $P$ is `new_basis`. 

We can rewrite this operation as:

$$ (x - \bar{x})P = xP - \bar{x}P = xP - \bar{x}_{\text{proj}} $$ 
$$ \bar{x}_{\text{proj}} = \bar{x}P $$

Why might we want to do this? In the real world, we want our data to take up as little storage as possible. Instead of storing a length $n$ vector $\bar{x}$, we can precompute $ \bar{x}_{\text{proj}} \in \mathbb{R}^3$ and store that instead!

Compute $ \bar{x}_{\text{proj}} $ using the **same mean vector** as the one computed with the training data.

In [ ]:
projected_mean_vec = # YOUR CODE HERE
print(new_basis.shape)
print(mean_vec)

Project the test data onto the **same PCA basis** as the one computed with the training data.

In [ ]:
projected_A_test = # YOUR CODE HERE

Zero-mean the projected test data using the **`projected_mean_vec`**.

In [ ]:
proj_demeaned = # YOUR CODE HERE
print(projected_mean_vec.shape)

Plot the projections to see how well your test data clusters in this new basis. This will give you an idea of test classification accuracy.

In [ ]:
if new_basis.shape[1] == 3:
    fig=plt.figure(figsize=(10,7))
    ax = fig.add_subplot(111, projection='3d')
    for i in range(len(all_words_arr)):
        Axes3D.scatter(ax, *proj_demeaned[i*num_samples_test:num_samples_test*(i+1)].T, c=cm[i], marker = 'o', s=20)
    plt.legend(all_words_arr,loc='center left', bbox_to_anchor=(1.07, 0.5))
    plt.title("Test Data")
    for i in range(len(all_words_arr)):
        Axes3D.scatter(ax, *np.array([centroids[i]]).T, c=cm[i], marker = '*', s=300)
    
    fig, axs = plt.subplots(1, 3, figsize=(15,5))
    for i in range(len(all_words_arr)):
        axs[0].scatter(proj_demeaned[i*num_samples_test:num_samples_test*(i+1),0], proj_demeaned[i*num_samples_test:num_samples_test*(i+1),1], c=cm[i], edgecolor='none')
        axs[1].scatter(proj_demeaned[i*num_samples_test:num_samples_test*(i+1),0], proj_demeaned[i*num_samples_test:num_samples_test*(i+1),2], c=cm[i], edgecolor='none')
        axs[2].scatter(proj_demeaned[i*num_samples_test:num_samples_test*(i+1),1], proj_demeaned[i*num_samples_test:num_samples_test*(i+1),2], c=cm[i], edgecolor='none')
    axs[0].set_title("View 1")
    axs[1].set_title("View 2")
    axs[2].set_title("View 3")
    plt.legend(all_words_arr, loc='center left', bbox_to_anchor=(1, 0.5))
    axs[0].scatter(centroid_list[:,0], centroid_list[:,1], c=colors, marker='*', s=300)
    axs[1].scatter(centroid_list[:,0], centroid_list[:,2], c=colors, marker='*', s=300)
    axs[2].scatter(centroid_list[:,1], centroid_list[:,2], c=colors, marker='*', s=300)
    fig.show()

elif new_basis.shape[1] == 2:
    fig=plt.figure(figsize=(10,7))
    for i in range(len(all_words_arr)):
        plt.scatter(proj_demeaned[i*num_samples_test:num_samples_test*(i+1),0], proj_demeaned[i*num_samples_test:num_samples_test*(i+1),1], c=colors[i], edgecolor='none')

    plt.scatter(centroid_list[:,0], centroid_list[:,1], c=colors, marker='*', s=300)
    plt.legend(all_words_arr,loc='center left', bbox_to_anchor=(1, 0.5))
    plt.title("Test Data")
    plt.show()

Now that we have some idea of how our test data looks in our PCA basis, let's see how our data actually performs. Implement the `classify` function which takes in a data point after enveloping is applied and returns which word number it belongs to depending on the closed centroid in Euclidian distance.

In [ ]:
def classify(word_set, data_point, new_basis, projected_mean_vec, centroids):
    """Classifies a new voice recording into a word.
    
    Args:
        word_set (list): set of words that we've chosen
        data_point (np.array): new data point vector before demeaning and projection
        new_basis (np.array): the new processed basis to project on
        projected_mean_vec (np.array): the same projected_mean_vec as before
    Returns:
        (string): The classified word
    Hint:
        Remember to use 'projected_mean_vec'!
        np.argmin(), and np.linalg.norm() may also help!
    """
    # TODO: classify the demeaned data point by comparing its distance to the centroids
    # YOUR CODE HERE
    projected_data_point = # YOUR CODE HERE
    
    # hint: demean with the precomputed mean vector
    demeaned = # YOUR CODE HERE 
    
    # hint: we're returning the word that this data point classified as
    return # YOUR CODE HERE

Try out the classification function below.

In [ ]:
# Try out the classification function
print(classify(all_words_arr, processed_A_test[0,:], new_basis, projected_mean_vec, centroids)) # Modify the row index of processed_A_test to use other vectors

**Our goal is 80% accuracy for each word.** Apply the `classify` function to each sample and compute the accuracy for each word. If you do not meet 80% accuracy for each word, try to find different combinations of words/parameters that result in more distinct clusters in the plots of the projected data.

In [ ]:
# Try to classify the whole A matrix
correct_counts = np.zeros(len(all_words_arr))

for (row_num, data) in enumerate(processed_A_test):
    word_num = row_num // num_samples_test
    if classify(all_words_arr, data, new_basis, projected_mean_vec, centroids) == all_words_arr[word_num]:
        correct_counts[word_num - 1] += 1
        
for i in range(len(correct_counts)):
    print("Percent correct of word {} = {}%".format(all_words_arr[i], 100 * correct_counts[i] / num_samples_test))

**<span style="color:red">PLEASE READ</span>** 

**Once you finish Task 5 with satisfactory accuracies, please come to the Cory 111 lab to try out your classifier!**

We recommend you go through the following process before going to Task 6.
1. Log onto the lab computers and open the Datahub link
2. Download your completed notebook
3. Download the zip link from the 16A website
4. Extract all files
5. Delete the existing voice_recognition_1_sp26.ipynb from the folder
6. Move your completed notebook to this folder
7. Open the Launch Notebook.bat to open up the notebook.

<a id='task6'></a>
## <span style="color:navy">Task 6: Testing the classifier -- Real Time</span>



Now, we'll be testing the classifier in real time. Run the script below, and when prompted, say one of your chosen words.

NOTE: Do not worry if your detector does not work as intended for some words. If you decided to record your own word in Task 1, it is likely your classifier will only detect your own word. This is likely due to the fact that your recording will have different frequencies compared to the recordings we provide.

In [ ]:
rate = 5400  # Sample rate
chunk = 1024  # Chunk size
record_seconds = 3  # Record duration in seconds
num_recordings = 50  # Total number of recordings needed
recording_count = 0
word = ""

while True:
    user_input = input("Press Enter to start recording, or type 'stop' and then Enter to stop recording: ")

    if user_input == '':
        # Record audio
        audio_recording = utils.record_audio(seconds=record_seconds, rate=rate, chunk=chunk)

        # Preprocess the single data. Hint: align_recording() might be useful here! make sure to set envelope=True.
        processed_audio_recording = ... # YOUR CODE HERE
        
        # Classify
        print("Classified Word: " + ...) # YOUR CODE HERE
        time.sleep(2)
       
    if user_input == 'stop':
        display.clear_output()
        break

<a id='feedback'></a>
## Feedback
If you have any feedback to give the teaching staff about the course (lab content, staff, etc), you can submit it through this Google form. Responses are **fully anonymous** and responses are actively monitored to improve the labs and course. Completing this form is **not required**.

[Anyonymous feedback Google form](https://docs.google.com/forms/d/e/1FAIpQLSch7s4OhheMytSjo3qPfyBv9_U8IBZPm0Syi1qGaKPANGNHFw/viewform?usp=header)

*If you have a personal matter to discuss or need a response to your feedback, please contact <a href="mailto:eecs16a.lab@berkeley.edu">eecs16a.lab@berkeley.edu</a> and/or <a href="mailto:eecs16a@berkeley.edu">eecs16a@berkeley.edu</a>*.

<a id='checkoff'></a>
## Checkoff
To receive credit, all labs will require the submission of a checkoff Google form. This link will be at the bottom of each lab. If you worked with a partner, both partners should fill out the form (you should have one submission per person), and feel free to use the same Google account/computer to fill it out as long as you have the correct names and student IDs.
If you completed the lab online, you must come to Cory 111 at a lab station first before filling out this form. Please do not fill out this form when you are not in the lab.

[Fill out the checkoff Google form.](https://tinyurl.com/16a-checkoff-sp26)

Your GSI or a Lab Assistant will come to your desk, check your final autograder result and ask some questions to test your understanding. Please be patient!